# 02 — Baselines: TF-IDF + Logistic Regression

**Question:** how well can a simple, fast, interpretable model predict the two targets from the review text alone?

* `recommendation_status`: `no_idea`, `not_recommended`, `recommended`
* `rating_band` (derived from the numeric rating): `low` (1 ≤ rate < 2.5), `medium` (2.5 ≤ rate < 4.0), `high` (4.0 ≤ rate ≤ 5.0)

This notebook **reads the saved results** of the baseline run (it does not retrain on the full data, which takes about 25 minutes).
The training code is `src/train_tfidf_baseline.py`; the test evaluation is `src/evaluate_test.py`. Sections 6 and 7 contain small live demonstrations.

Rules followed: the product-grouped split (`SPLIT_SEED = 2026`), vocabulary and weights fitted on training rows only, the numeric rating is never an input,
the test set is evaluated once (section 4).

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
REPORTS = ROOT / "reports"
ARTIFACTS = ROOT / "artifacts"
PROCESSED = ROOT / "data" / "processed"
REC = ["no_idea", "not_recommended", "recommended"]
BAND = ["low", "medium", "high"]
SEED = 2026
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

val_rec = load_json(REPORTS / "baseline_tfidf_word_recommendation_status_validation.json")
val_band = load_json(REPORTS / "baseline_tfidf_word_rating_band_validation.json")
print("project root:", ROOT)

## 1. Method

| part | choice |
|---|---|
| input | normalized review text (`body_norm`); the numeric rating is never used |
| features | TF-IDF on words, unigrams and bigrams, `min_df=3`, at most 300,000 features, `sublinear_tf=True`, one-letter tokens kept (`token_pattern=r"(?u)\b\w+\b"`) |
| model | multinomial logistic regression, `C=1.0`, `class_weight="balanced"`, `lbfgs`, at most 1000 iterations |
| two targets | two separate models, each with its own vectorizer, fitted only on `train_supervised` rows that have that target |
| split | product-grouped 70/15/15, seed 2026; masked rows (the artificial unlabeled pool) are not used |
| evaluation | Macro F1 first (the classes are imbalanced), plus accuracy, balanced accuracy, per-class precision/recall/F1, confusion matrix; for `rating_band` also the ordinal distance error |

`class_weight="balanced"` gives rare classes (`no_idea`, `not_recommended`, `low`) a larger weight, so the model does not only predict the big class.

In [ ]:
cfg = pd.DataFrame([
    {"head": "recommendation", "train rows": val_rec["train_rows"], "validation rows": val_rec["validation_rows"],
     "features": val_rec["n_features"], "iterations used": f'{val_rec["iterations_used"]} of {val_rec["max_iter"]}',
     "converged": val_rec["converged"]},
    {"head": "rating band", "train rows": val_band["train_rows"], "validation rows": val_band["validation_rows"],
     "features": val_band["n_features"], "iterations used": f'{val_band["iterations_used"]} of {val_band["max_iter"]}',
     "converged": val_band["converged"]},
])
cfg

## 2. Experiments on a 300k-row sample: words or character n-grams?

Hypothesis: because spelling in reviews is inconsistent (stretched letters, typos), character n-grams might be more robust than words.
Both were trained on the same sample (seed 2026) and compared on the same validation sample, recommendation head only.

**Provenance:** the numbers below were printed by the sample experiments run earlier (`03_baselines.ipynb`, one run each) and are copied here; they are not recomputed in this notebook.

In [ ]:
sample_exp = pd.DataFrame([
    {"features": "word 1-2 grams", "train rows": 248915, "validation rows": 85327, "macro F1": 0.6551, "balanced acc": 0.7071,
     "F1 no_idea": 0.411, "F1 not_recommended": 0.658, "F1 recommended": 0.896, "n features": 111933, "matrix build (s)": 9.3, "fit (s)": 27.2},
    {"features": "char 2-5 grams (char_wb)", "train rows": 248915, "validation rows": 85327, "macro F1": 0.6338, "balanced acc": 0.6994,
     "F1 no_idea": 0.393, "F1 not_recommended": 0.632, "F1 recommended": 0.877, "n features": 75109, "matrix build (s)": 28.1, "fit (s)": 32.7},
])
sample_exp

**Reading:** in this setting character n-grams were **not better** (about 0.02 lower macro F1, three times slower to build).
This is one run on one sample with untuned settings, so we only conclude "not better here", not "words are always better". Words were kept for the baseline.

## 3. Final baseline on the validation set, and how much does more training data help?

In [ ]:
def head_table(js, labels):
    rows = [{"metric": "macro F1", "value": js["macro_f1"]}, {"metric": "balanced accuracy", "value": js["balanced_accuracy"]}]
    rows += [{"metric": f"F1 {l}", "value": js["f1_per_class"][l]} for l in labels]
    if "ordinal_distance_error" in js:
        rows += [{"metric": "ordinal distance error", "value": js["ordinal_distance_error"]},
                 {"metric": "share of distance-2 errors", "value": js["share_distance_2_errors"]}]
    return pd.DataFrame(rows).set_index("metric")

pd.concat({"recommendation": head_table(val_rec, REC), "rating band": head_table(val_band, BAND)}, axis=1).round(4)

In [ ]:
def plot_confusions(js_list, titles, labels_list):
    fig, axes = plt.subplots(1, len(js_list), figsize=(6 * len(js_list), 4.6))
    axes = np.atleast_1d(axes)
    for ax, js, title, labels in zip(axes, js_list, titles, labels_list):
        cm = np.array(js["confusion_matrix"]["rows_true_cols_pred"], dtype=float)
        share = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=20)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{share[i, j]:.0%}\n{int(cm[i, j]):,}", ha="center", va="center",
                        color="white" if share[i, j] > 0.5 else "black", fontsize=9)
    plt.tight_layout(); plt.show()

plot_confusions([val_rec, val_band], ["recommendation (validation)", "rating band (validation)"], [REC, BAND])

Each row is a true class; the cell shows the share of that class predicted as each column (and the count).
Typical pattern: the big classes (`recommended`, `high`) are predicted well; `no_idea` and `medium` are confused with their neighbours.

In [ ]:
# Effect of the amount of training data (macro F1). The 1M-row run is read from its saved json;
# the 300k-row values were printed earlier (same notebook as section 2) and are copied.
def f1_of(path, default):
    p = Path(path)
    return load_json(p)["macro_f1"] if p.exists() else default

curve = pd.DataFrame([
    {"head": "recommendation", "train rows": 248915, "evaluated on": "validation sample (85k)", "macro F1": 0.6551},
    {"head": "recommendation", "train rows": 829591, "evaluated on": "full validation", "macro F1": f1_of(REPORTS / "baseline_1M_recommendation_status_validation.json", np.nan)},
    {"head": "recommendation", "train rows": val_rec["train_rows"], "evaluated on": "full validation", "macro F1": val_rec["macro_f1"]},
    {"head": "rating band", "train rows": 275975, "evaluated on": "validation sample (91k)", "macro F1": 0.6213},
    {"head": "rating band", "train rows": 919179, "evaluated on": "full validation", "macro F1": f1_of(REPORTS / "baseline_1M_rating_band_validation.json", np.nan)},
    {"head": "rating band", "train rows": val_band["train_rows"], "evaluated on": "full validation", "macro F1": val_band["macro_f1"]},
])
display(curve)
fig, ax = plt.subplots(figsize=(6, 3.8))
for h, g in curve.groupby("head"):
    ax.plot(g["train rows"], g["macro F1"], "o-", label=h)
ax.set_xscale("log"); ax.set_xlabel("training rows (log scale)"); ax.set_ylabel("validation macro F1"); ax.grid(alpha=.3); ax.legend()
plt.show()

**Reading:** about 3.5 times more data gave only about +0.005 to +0.006 macro F1 at the end of the curve. The simple model is limited by its representation, not by data size
(this is an interpretation; it is not proven here). The first point of each head was evaluated on a validation *sample*, so only the last two points are strictly comparable.

## 4. Test set (evaluated once)

`src/evaluate_test.py --model tfidf --name tfidf_baseline` evaluated this model once on a fixed random test sample (seed 2026) that every later model also uses.
If the result file is missing, run that command first.

In [ ]:
test_path = REPORTS / "test_results" / "tfidf_baseline.json"
if test_path.exists():
    T = load_json(test_path)
    def test_table(m, labels):
        rows = {"rows": m["n"], "accuracy": m["accuracy"], "balanced accuracy": m["balanced_accuracy"], "macro precision": m["macro_precision"],
                "macro recall": m["macro_recall"], "macro F1": m["macro_f1"], "ECE (calibration)": m["ece"]}
        if "ordinal_distance_error" in m:
            rows.update({"ordinal distance error": m["ordinal_distance_error"], "share of distance-2 errors": m["share_distance_2_errors"],
                         "quadratic weighted kappa": m["quadratic_weighted_kappa"]})
        t1 = pd.Series(rows).to_frame("value")
        t2 = pd.DataFrame(m["per_class"]).T[["precision", "recall", "f1", "support"]]
        return t1, t2
    for key, labels, name in (("rec", REC, "recommendation"), ("band", BAND, "rating band")):
        t1, t2 = test_table(T[key], labels)
        print(f"===== {name} (test) =====")
        display(t1.round(4)); display(t2.round(3))
    plot_confusions([{"confusion_matrix": {"rows_true_cols_pred": T["rec"]["confusion_matrix"]}},
                     {"confusion_matrix": {"rows_true_cols_pred": T["band"]["confusion_matrix"]}}],
                    ["recommendation (test)", "rating band (test)"], [REC, BAND])
else:
    print("test result file not found:", test_path)

**Validation versus test:** the macro F1 on the test sample is almost the same as on the full validation set (recommendation 0.6728 and 0.6728; rating band 0.6364 and 0.6357),
so the split is consistent and the numbers did not depend on which of the two sets we looked at.

## 5. What did the model learn? (interpretability)

For a linear model the weights can be read directly: the terms with the largest positive weight for a class push the prediction towards that class.

In [ ]:
def top_terms(file_part, labels, k=12):
    vec = joblib.load(ARTIFACTS / f"tfidf_word_{file_part}_vectorizer.joblib")
    clf = joblib.load(ARTIFACTS / f"tfidf_word_{file_part}_logreg.joblib")
    names = vec.get_feature_names_out()
    cols = {}
    for lab in labels:
        i = list(clf.classes_).index(lab)
        cols[lab] = [names[j] for j in np.argsort(clf.coef_[i])[::-1][:k]]
    return pd.DataFrame(cols)

print("recommendation: highest-weight terms per class"); display(top_terms("recommendation_status", REC))
print("rating band: highest-weight terms per class"); display(top_terms("rating_band", BAND))

Read the lists with care: a term can have a high weight because it is frequent in that class, not because it is the *reason* for the label, and bigrams overlap with their words.
A linear model cannot see word order beyond bigrams, which is why it struggles with negation scope and mixed opinions (see the error analysis in the report).

## 6. Live demonstration of the same pipeline on a 100k-row sample

This repeats the baseline code on a small sample in this notebook (about a minute). It is a demonstration only: nothing is saved and the numbers are not the final baseline.

In [ ]:
RUN_DEMO = True
if RUN_DEMO:
    import time
    import duckdb
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import f1_score, accuracy_score, classification_report

    con = duckdb.connect()
    def sample(name, n):
        path = (PROCESSED / name).as_posix()
        return con.execute(f"""
            SELECT body_norm, recommendation_status AS y
            FROM read_csv('{path}', header=true, all_varchar=true)
            WHERE body_norm IS NOT NULL AND recommendation_status IS NOT NULL
            USING SAMPLE reservoir({n} ROWS) REPEATABLE ({SEED})""").df()

    t0 = time.time()
    tr, va = sample("train_supervised.csv", 100000), sample("validation.csv", 20000)
    vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3, max_features=300000,
                          sublinear_tf=True, token_pattern=r"(?u)\b\w+\b")
    X_tr, X_va = vec.fit_transform(tr["body_norm"]), vec.transform(va["body_norm"])   # vocabulary from training rows only
    clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=300).fit(X_tr, tr["y"])
    pred = clf.predict(X_va)
    print(f"train {len(tr):,} rows, validation {len(va):,} rows, {time.time() - t0:.0f} s")
    print("macro F1:", round(f1_score(va["y"], pred, average="macro"), 4), "| accuracy:", round(accuracy_score(va["y"], pred), 4))
    print(classification_report(va["y"], pred, digits=3))

## 7. Conclusions and limits

1. **Baseline:** TF-IDF + logistic regression reaches macro F1 **0.6728** (recommendation) and **0.6357** (rating band) on validation and 0.6728 / 0.6364 on the test sample. It trains in minutes and its weights can be inspected.
2. **Hardest classes:** `no_idea` (recommendation) and `medium` (rating band) are the weakest in every experiment: their texts are close to both neighbouring classes.
3. **Words versus characters:** characters were not better in one sample experiment (about 0.02 lower macro F1); only one run, settings not tuned.
4. **More data helps little:** about +0.005 to +0.006 macro F1 for 3.5 times more data at the end of the curve; the limit is the representation (bag of words and bigrams), which motivates the fine-tuned Transformer in `03_transfer_learning_finetuning.ipynb`.
5. **Limits:** `C` and the vocabulary size were not tuned; calibration was measured (ECE 0.029 and 0.055 on test) but not improved; the test numbers are from a fixed 99,635-row sample, not the whole test file.

The Transformer comparison with paired bootstrap confidence intervals is in `03_transfer_learning_finetuning.ipynb`.